In [ ]:
import sys
print(sys.executable)

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv("statement.csv")

# Display the first few rows
df.head()

In [ ]:
# Function to extract text between first and second '/'
def extract_between_slashes(text):
    if pd.isna(text):  # Handle NaN(missing) values
        return 'undefined'
    
    text = str(text)  # Convert to string
    parts = text.split('/')
    
    # Check if there are at least 2 slashes (meaning 3 parts)
    if len(parts) >= 3:
        # Replace \r with space and strip extra whitespace
        return parts[1].replace('\r', ' ').strip()
    else:
        return 'undefined'

# Apply the function to Description column
df['merchant_hint'] = df['Description'].apply(extract_between_slashes)

# Display the result
df[['Description', 'merchant_hint']].head(100)
# The code only modifies the DataFrame in memory. The original statement.csv file remains unchanged.

In [ ]:
# Group by merchant_hint and count occurrences
merchant_counts = df['merchant_hint'].value_counts()
print(f"Total unique merchants: {len(merchant_counts)}\n")
print(merchant_counts)

In [ ]:
# Get unique merchant hints (excluding 'undefined')
unique_merchants = df['merchant_hint'][df['merchant_hint'] != 'undefined'].unique()
print(f"\nMerchants to categorize: {len(unique_merchants)}")
print(list(unique_merchants))

In [ ]:
import requests, json
 # Function starts here
def categorize_merchant(merchant_name):
    prompt = f"""You are a financial transaction categorizer. Categorize this merchant into the MOST APPROPRIATE category.

Categories and examples:
- Food & Dining: restaurants, Swiggy, Zomato, cafes, food delivery, dineout
- Groceries: supermarkets, BigBasket, Blinkit, grocery stores, zeptonow, zepto
- Shopping: retail stores, Amazon, Flipkart, clothing, electronics
- Entertainment: movies, games, events, bookmyshow
- Subscriptions: Netflix, spotify, Prime, Apple Music, YouTube Premium, Openai, openai llc
- Travel: Uber, Ola, fuel, parking, flights, indigo, air india
- Bills & Utilities: electricity, water, internet, phone bills, airtel, jio, reliance j, google
- P2P Transfer: person names, UPI transfers to individuals, containing '@' , containing 'paytm','gpay'
- Healthcare: hospitals, pharmacies
- Self Transfer: if it contains ( 'Harsh', 'Harshs', 'HARSH SING' or 'Harshsingla') .. nothing else is self transfer
- Other: anything else

Merchant name: {merchant_name}

Think step by step:
1. What type of service does this merchant provide?
2. Which category fits best?

Answer with ONLY the category name from the list above. No explanation."""
#try is Python's error handling mechanism. It lets you attempt code that might fail without crashing your entire program.
    try:
        response = requests.post(
            "http://localhost:11434/api/chat",
            json={
                "model": "llama3.2:3b",
                "messages": [{"role": "user", "content": prompt}], # Sends your prompt as a user message
                "stream": False, # Get complete response at once (not word-by-word)
                "options": {"temperature": 0.1}
            }
        )
        
        category = response.json()["message"]["content"].strip()
        
        # Clean up the response
        valid_categories = [
            'Food & Dining', 'Groceries', 'Shopping', 'Entertainment',
            'Subscriptions', 'Travel', 'Bills & Utilities',
            'P2P Transfer', 'Healthcare', 'Other', 'Self Transfer'
        ]
        
        for valid_cat in valid_categories:
            if valid_cat.lower() in category.lower():
                return valid_cat
        #Loops through each valid category
        #Checks if any valid category name appears in the AI's response (case-insensitive)
        #Returns the first match found
        #Why needed? The AI might respond with extra text like "The category is: Food & Dining" instead of just "Food & Dining"
              
        return category
    except Exception as e:
        print(f"Error categorizing {merchant_name}: {e}")
        return 'Other'
# Function ends here
   


In [ ]:
# Test with one merchant first
test_merchant = unique_merchants[0]
category = categorize_merchant(test_merchant)
print(f"{test_merchant} -> {category}")

In [ ]:
# Create a dictionary to store merchant categories
merchant_categories = {}

print("Categorizing merchants...\n")

for i, merchant in enumerate(unique_merchants, 1): # with enumerate you get both the item AND its position
    category = categorize_merchant(merchant)
    merchant_categories[merchant] = category
    print(f"{i}. {merchant} -> {category}")

In [ ]:
print("Current columns:", df.columns.tolist())

In [ ]:
# Clean column names (prevents silent mismatches)
df.columns = df.columns.str.strip().str.strip("'\"")


In [ ]:
# Map categories back to the main dataframe
df['category'] = df['merchant_hint'].map(merchant_categories)

# For 'undefined' merchants, set category as 'other'
df.loc[df['merchant_hint'] == 'undefined', 'category'] = 'Other'

# Convert Amount to numeric (if not already)
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce') 
#errors='coerce' - if conversion fails, set to NaN (not a number) instead of crashing

# Create a signed amount column: DR is negative (expense), CR is positive (income)
df['signed_amount'] = df.apply(
    lambda row: -row['Amount'] if row['Type'] == 'DR' else row['Amount'], 
    axis=1
)

print("\n" + "="*60)
print("CATEGORIZATION COMPLETE!")
print("="*60 + "\n")

# Show summary by category with proper DR/CR handling
print("Summary by Category (DR = Expense, CR = Income):")
category_summary = df.groupby('category').agg({
    'signed_amount': 'sum',
    'Amount': 'count'
}).round(2)
category_summary.columns = ['Net Amount', 'Transaction Count']
category_summary = category_summary.sort_values('Net Amount')
print(category_summary)

print("\n" + "="*60 + "\n")

# Total spent vs received
total_spent = df[df['Type'] == 'DR']['Amount'].sum()
total_received = df[df['Type'] == 'CR']['Amount'].sum()
net_balance = total_received - total_spent

print(f"Total Spent (DR):     ₹{total_spent:,.2f}")
print(f"Total Received (CR):  ₹{total_received:,.2f}")
print(f"Net Balance:          ₹{net_balance:,.2f}")

print("\n" + "="*60 + "\n")

# Show detailed breakdown
print("Sample transactions with categories:")
df[['Date', 'merchant_hint', 'Amount', 'Type', 'signed_amount', 'category']].head(20)

In [ ]:
# Check the actual column names
print("Columns:", df.columns.tolist())
print("Column types:", [type(col) for col in df.columns])

# Fix: Strip any whitespace and quotes from column names
df.columns = df.columns.str.strip().str.strip("'\"")

# Verify the fix
print("\nFixed columns:", df.columns.tolist())

# Ensure derived columns exist (guards against out-of-order runs)
if 'merchant_hint' not in df.columns:
    df['merchant_hint'] = df['Description'].apply(extract_between_slashes)

if 'category' not in df.columns:
    df['category'] = df['merchant_hint'].map(merchant_categories)
    df.loc[df['merchant_hint'] == 'undefined', 'category'] = 'Other'

print("Columns just before save:", df.columns.tolist())
assert 'merchant_hint' in df.columns, "merchant_hint missing before save"
assert 'category' in df.columns, "category missing before save"
 
# Now save again
preferred_order = ['Date', 'Description', 'merchant_hint', 'category', 'Amount', 'Type', 'month', 'signed_amount']
cols_to_save = [c for c in preferred_order if c in df.columns]
df.to_csv("statement_categorized.csv", columns=cols_to_save, index=False)

print("✅ Categories saved to 'statement_categorized.csv'")

# Verify what was saved
df_test = pd.read_csv("statement_categorized.csv")
print("\nColumns in saved file:", df_test.columns.tolist())
print(f"Total rows: {len(df_test)}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# Load data
df = pd.read_csv("statement_categorized.csv")

# Prepare data
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['month'] = df['Date'].dt.strftime('%Y-%m')
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

# Filter out self-transfers
spending_df = df[(df['Type'] == 'DR') & (df['category'] != 'Self Transfer')]

print("🎨 Generating charts...\n")

# Create subplots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Financial Overview Dashboard', fontsize=16, fontweight='bold')

# ============ CHART 1: Monthly Spending Trend ============
monthly_spending = spending_df.groupby('month')['Amount'].sum().sort_index()
months_labels = [datetime.strptime(m, '%Y-%m').strftime('%b %y') for m in monthly_spending.index]

ax1.plot(months_labels, monthly_spending.values, marker='o', linewidth=2, markersize=8, color='#3b82f6')
ax1.fill_between(range(len(monthly_spending)), monthly_spending.values, alpha=0.3, color='#3b82f6')
ax1.set_title('Monthly Spending Trend', fontweight='bold', fontsize=12)
ax1.set_xlabel('Month')
ax1.set_ylabel('Amount (₹)')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)

# Add values on top of points
for i, v in enumerate(monthly_spending.values):
    ax1.text(i, v, f'₹{int(v/1000)}k', ha='center', va='bottom', fontsize=9)

# ============ CHART 2: Category Distribution ============
category_spending = spending_df.groupby('category')['Amount'].sum().sort_values(ascending=False)
colors = plt.cm.Set3(range(len(category_spending)))

wedges, texts, autotexts = ax2.pie(
    category_spending.values, 
    labels=category_spending.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90
)
ax2.set_title('Spending by Category', fontweight='bold', fontsize=12)

# Make percentage text more readable
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(9)

# ============ CHART 3: Top 10 Merchants ============
top_merchants = spending_df.groupby('merchant_hint')['Amount'].sum().sort_values(ascending=False).head(10)

# Shorten merchant names if too long
merchant_labels = [m[:20] + '...' if len(m) > 20 else m for m in top_merchants.index]

bars = ax3.barh(range(len(top_merchants)), top_merchants.values, color='#10b981')
ax3.set_yticks(range(len(top_merchants)))
ax3.set_yticklabels(merchant_labels, fontsize=9)
ax3.set_title('Top 10 Merchants', fontweight='bold', fontsize=12)
ax3.set_xlabel('Amount (₹)')
ax3.invert_yaxis()

# Add values on bars
for i, (bar, value) in enumerate(zip(bars, top_merchants.values)):
    ax3.text(value, i, f' ₹{int(value)}', va='center', fontsize=9)

# ============ CHART 4: Category Comparison (Bar) ============
category_spending_sorted = category_spending.sort_values(ascending=True)
bars = ax4.barh(range(len(category_spending_sorted)), category_spending_sorted.values, color='#f59e0b')
ax4.set_yticks(range(len(category_spending_sorted)))
ax4.set_yticklabels(category_spending_sorted.index, fontsize=9)
ax4.set_title('All Categories Comparison', fontweight='bold', fontsize=12)
ax4.set_xlabel('Amount (₹)')

# Add values on bars
for i, (bar, value) in enumerate(zip(bars, category_spending_sorted.values)):
    ax4.text(value, i, f' ₹{int(value)}', va='center', fontsize=9)

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()

# ============ PRINT SUMMARY STATS ============
print("\n" + "="*50)
print("📊 SUMMARY STATISTICS")
print("="*50)

total_spent = spending_df['Amount'].sum()
avg_transaction = spending_df['Amount'].mean()
transaction_count = len(spending_df)

months_sorted = sorted(spending_df['month'].unique())
if len(months_sorted) >= 2:
    last_month = months_sorted[-1]
    prev_month = months_sorted[-2]
    last_month_spending = spending_df[spending_df['month'] == last_month]['Amount'].sum()
    prev_month_spending = spending_df[spending_df['month'] == prev_month]['Amount'].sum()
    change = ((last_month_spending - prev_month_spending) / prev_month_spending * 100) if prev_month_spending > 0 else 0
    
    print(f"\n💰 Total Spent: ₹{total_spent:,.2f}")
    print(f"📊 Average Transaction: ₹{avg_transaction:,.2f}")
    print(f"📝 Total Transactions: {transaction_count}")
    print(f"\n📅 Last Month: {datetime.strptime(last_month, '%Y-%m').strftime('%B %Y')}")
    print(f"   Spent: ₹{last_month_spending:,.2f}")
    print(f"   Change: {change:+.1f}% vs previous month")
    
    print(f"\n🏆 Top Category: {category_spending.index[0]}")
    print(f"   Amount: ₹{category_spending.values[0]:,.2f}")
    
    print(f"\n🏪 Top Merchant: {top_merchants.index[0]}")
    print(f"   Amount: ₹{top_merchants.values[0]:,.2f}")

print("\n" + "="*50)
print("✅ Charts displayed!")
print("="*50)

In [ ]:
import pandas as pd
import requests
from datetime import datetime
import gradio as gr

# Load data
df = pd.read_csv("statement_categorized.csv")
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['month'] = df['Date'].dt.strftime('%Y-%m')
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

# Filter out self-transfers
spending_df = df[(df['Type'] == 'DR') & (df['category'] != 'Self Transfer')]

print("✅ Data loaded successfully!")
print(f"📊 Total transactions: {len(df)}")
print(f"💸 Actual expenses: {len(spending_df)}")

# ========= MATPLOTLIB CHART FUNCTIONS =========

import matplotlib.pyplot as plt
import io
from PIL import Image

def chart_monthly_trend():
    monthly = spending_df.groupby('month')['Amount'].sum().sort_index()

    fig, ax = plt.subplots(figsize=(7, 4), dpi=150)  # ⬅ Bigger + sharper
    ax.plot(monthly.index, monthly.values, marker="o")
    ax.set_title("Monthly Spending Trend")
    ax.set_xlabel("Month")
    ax.set_ylabel("Amount (₹)")
    ax.grid(True, alpha=0.3)

    plt.xticks(rotation=45, ha='right')  # ⬅ prevent merging

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    buf.seek(0)
    plt.close(fig)
    return Image.open(buf)


def chart_category_distribution():
    cat = spending_df.groupby('category')['Amount'].sum().sort_values()

    fig, ax = plt.subplots(figsize=(7, 4), dpi=150)  # ⬅ Bigger
    ax.barh(cat.index, cat.values)
    ax.set_title("Category Distribution")
    ax.set_xlabel("Amount (₹)")
    ax.grid(True, alpha=0.3)

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    buf.seek(0)
    plt.close(fig)
    return Image.open(buf)


# ============ HELPER FUNCTIONS ============

def get_insights():
    """Generate quick insights"""
    insights = []
    
    months_sorted = sorted(spending_df['month'].unique())
    if len(months_sorted) == 0:
        return "No spending data available"
    
    last_month = months_sorted[-1]
    last_month_name = datetime.strptime(last_month, '%Y-%m').strftime('%B %Y')
    last_month_spending = spending_df[spending_df['month'] == last_month]['Amount'].sum()
    
    insights.append(f"📅 Latest: {last_month_name}")
    insights.append(f"💰 Spent: ₹{last_month_spending:,.0f}")
    
    if len(months_sorted) >= 2:
        prev_month = months_sorted[-2]
        prev_spending = spending_df[spending_df['month'] == prev_month]['Amount'].sum()
        change = ((last_month_spending - prev_spending) / prev_spending * 100) if prev_spending > 0 else 0
        insights.append(f"📈 Change: {change:+.1f}% vs previous month")
    
    top_cat = spending_df[spending_df['month'] == last_month].groupby('category')['Amount'].sum().nlargest(1)
    if len(top_cat) > 0:
        insights.append(f"🏆 Top category: {top_cat.index[0]} (₹{top_cat.iloc[0]:,.0f})")
    
    total_spent = spending_df['Amount'].sum()
    insights.append(f"💵 Total spent: ₹{total_spent:,.0f}")
    
    return "\n".join(insights)


def query_ollama(question):
    """Query Ollama AI"""
    
    # Get data
    total_spent = spending_df['Amount'].sum()
    total_received = df[df['Type'] == 'CR']['Amount'].sum()
    
    # Monthly spending
    monthly_spending = spending_df.groupby('month')['Amount'].sum().sort_index()
    monthly_text = "\n".join([
        f"  {datetime.strptime(m, '%Y-%m').strftime('%B %Y')}: ₹{amt:,.0f}"
        for m, amt in monthly_spending.items()
    ])
    
    # Categories
    category_spending = spending_df.groupby('category')['Amount'].sum().sort_values(ascending=False)
    category_text = "\n".join([
        f"  {i}. {cat}: ₹{amt:,.0f}"
        for i, (cat, amt) in enumerate(category_spending.items(), 1)
    ])
    
    # Top 10 merchants
    top_merchants = spending_df.groupby('merchant_hint')['Amount'].sum().sort_values(ascending=False).head(10)
    merchant_text = "\n".join([
        f"  {i}. {merchant}: ₹{amt:,.0f}"
        for i, (merchant, amt) in enumerate(top_merchants.items(), 1)
    ])
    
    # Create prompt
    system_prompt = f"""You are a financial assistant analyzing a bank statement.

FINANCIAL DATA (excludes self-transfers):

SUMMARY:
  Total Spent: ₹{total_spent:,.0f}
  Total Received: ₹{total_received:,.0f}
  Net Balance: ₹{(total_received - total_spent):,.0f}

MONTHLY SPENDING:
{monthly_text}

SPENDING BY CATEGORY (ranked):
{category_text}

TOP 10 MERCHANTS (ranked):
{merchant_text}

IMPORTANT RULES:
1. Use EXACT numbers from the data above
2. #1 means FIRST in the list (highest)
3. Format amounts as ₹X,XXX
4. Be brief (2-3 sentences)
5. NEVER guess or make up numbers

USER QUESTION: {question}

Your answer:"""

    try:
        response = requests.post(
            "http://localhost:11434/api/chat",
            json={
                "model": "llama3.2:3b",
                "messages": [{"role": "user", "content": system_prompt}],
                "stream": False,
                "options": {
                    "temperature": 0.1,
                    "num_predict": 200,
                    "num_ctx": 8192
                }
            },
            timeout=30
        )
        
        if response.status_code != 200:
            return f"❌ Ollama error (status {response.status_code})"
        
        return response.json()["message"]["content"].strip()
        
    except requests.exceptions.ConnectionError:
        return "❌ Ollama not running. Start it with: ollama serve"
    except requests.exceptions.Timeout:
        return "⏱️ Request timed out. Try a simpler question."
    except Exception as e:
        return f"❌ Error: {str(e)}"


# ============ GRADIO CHATBOT ============

def chatbot_response(message, history):
    """Handle chatbot responses"""
    
    if not message.strip():
        return ""
    
    # Handle special commands
    if message.lower().strip() in ['insights', 'summary']:
        return get_insights()
    
    # Query AI
    response = query_ollama(message)
    return response


# ============ CREATE GRADIO UI ============

with gr.Blocks(theme=gr.themes.Soft(), title="Financial Assistant", css="""
    .gradio-container {
        max-width: 100% !important;
        padding: 20px !important;
    }
    footer {display: none !important;}
""") as demo:
    
    gr.Markdown("""
    # 💰 Financial Assistant Chatbot  
    ### Powered by Llama 3.2  
    Ask me anything about your spending!
    """)

    # ======= MAIN 3-COLUMN LAYOUT =======
    with gr.Row():
        
        # ✅ LEFT SIDE — CHATBOT (60% width)
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                height=700,
                label="Chat",
                show_label=False,
                avatar_images=(None, "🤖"),
                type="messages"
            )

            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Ask me about your spending...",
                    show_label=False,
                    scale=5
                )
                submit = gr.Button("Send 📤", scale=1, variant="primary")

        # ✅ MIDDLE PANEL — QUICK STATS + EXAMPLES (20% width)
        with gr.Column(scale=2):
            gr.Markdown("### 📊 Quick Stats")
            stats_display = gr.Textbox(
                value=get_insights(),
                show_label=False,
                lines=15,
                interactive=False
            )

            gr.Markdown("### 💡 Example Questions")
            examples = gr.Examples(
                examples=[
                    "How much did I spend in October?",
                    "What's my top spending category?",
                    "To which merchant I paid most?",
                    "Compare my last 3 months spending",
                    "Am I spending too much on food?",
                    "What's my spending trend?",
                ],
                inputs=msg
            )

            clear_btn = gr.Button("🔄 Clear Chat", variant="secondary")

        # ✅ RIGHT SIDE — CHARTS (30% width)
        with gr.Column(scale=3):
          
                gr.Markdown("### Spending Charts")

                trend_btn = gr.Button("Show Monthly Trend")
                trend_img = gr.Image(
                    label="Monthly Spending Trend",
                    width=900
                )

                cat_btn = gr.Button("Show Category Distribution")
                cat_img = gr.Image(
                    label="Category Distribution",
                    width=900
                )

                trend_btn.click(chart_monthly_trend, outputs=trend_img)
                cat_btn.click(chart_category_distribution, outputs=cat_img)

    # ======= FOOTER =======
    gr.Markdown("""
    ---
    **Tips:**
    - Type **'insights'** for a quick summary  
    - Ask natural questions about your spending  
    - The AI analyzes your statement in real-time  
    """)

    # Event handlers
    def respond(message, chat_history):
        bot_message = chatbot_response(message, chat_history)
        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": bot_message})
        return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    submit.click(respond, [msg, chatbot], [msg, chatbot])
    clear_btn.click(lambda: None, None, chatbot, queue=False)



# ============ LAUNCH ============

if __name__ == "__main__":
    print("\n" + "="*60)
    print("🚀 Launching Gradio Chatbot UI...")
    print("="*60 + "\n")
    
    demo.launch(
        share=False,  # Set to True to create a public link
        server_name="127.0.0.1",
        server_port=None,  # Auto-find available port
        show_error=True,
        inbrowser=True  # Automatically opens browser
    )